# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
from itertools import product

## 1. Preprocessing

In [2]:
# Load not-scaled data and add dayofweek from scaled version
df_ns = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_scaled = pd.read_csv('../data/dayofweek.csv')

df_ns['dayofweek'] = df_scaled['dayofweek'].values
print(df_ns.shape)
df_ns.head()

(1686, 44)


,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [3]:
X = df_ns.drop('dayofweek', axis=1).values
y = df_ns['dayofweek'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (1348, 43) Test: (338, 43)


## 2. SVM gridsearch

In [4]:
svm_params = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21],
    'probability': [True]
}

gs_svm = GridSearchCV(
    SVC(), param_grid=svm_params,
    scoring='accuracy', cv=5, n_jobs=-1, verbose=1
)
gs_svm.fit(X_train, y_train)
print('Best params:', gs_svm.best_params_)
print('Best CV score:', gs_svm.best_score_)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best CV score: 0.8761090458488228


In [12]:
svm_results = pd.DataFrame(gs_svm.cv_results_)
svm_results = svm_results[['params', 'mean_test_score', 'rank_test_score']]
svm_results = svm_results.sort_values('rank_test_score')
svm_results.head(10)

,params,mean_test_score,rank_test_score
70,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.876109,1
64,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.863500,2
58,"{'C': 5, 'class_weight': None, 'gamma': 'auto'...",0.816018,3
52,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.807865,4
63,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.721052,5
60,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.721052,5
66,"{'C': 10, 'class_weight': None, 'gamma': 'scal...",0.719587,7
69,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.719587,7
51,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.706234,9
48,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.706234,9


## 3. Decision tree

In [6]:
tree_params = {
    'max_depth': list(range(1, 50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

gs_tree = GridSearchCV(
    DecisionTreeClassifier(), param_grid=tree_params,
    scoring='accuracy', cv=5, n_jobs=-1, verbose=1
)
gs_tree.fit(X_train, y_train)
print('Best params:', gs_tree.best_params_)
print('Best CV score:', gs_tree.best_score_)

Fitting 5 folds for each of 196 candidates, totalling 980 fits
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 23, 'random_state': 21}
Best CV score: 0.8738592867960897


In [7]:
tree_results = pd.DataFrame(gs_tree.cv_results_)
tree_results = tree_results[['params', 'mean_test_score', 'rank_test_score']]
tree_results = tree_results.sort_values('rank_test_score')
tree_results.head(10)

,params,mean_test_score,rank_test_score
97,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
95,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
94,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
71,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
72,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
73,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
75,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
76,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
77,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
78,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1


## 4. Random forest

In [8]:
rf_params = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': list(range(1, 50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

gs_rf = GridSearchCV(
    RandomForestClassifier(), param_grid=rf_params,
    scoring='accuracy', cv=5, n_jobs=-1, verbose=1
)
gs_rf.fit(X_train, y_train)
print('Best params:', gs_rf.best_params_)
print('Best CV score:', gs_rf.best_score_)

Fitting 5 folds for each of 784 candidates, totalling 3920 fits
Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 28, 'n_estimators': 50, 'random_state': 21}
Best CV score: 0.9042902381935839


In [9]:
rf_results = pd.DataFrame(gs_rf.cv_results_)
rf_results = rf_results[['params', 'mean_test_score', 'rank_test_score']]
rf_results = rf_results.sort_values('rank_test_score')
rf_results.head(10)

,params,mean_test_score,rank_test_score
698,"{'class_weight': None, 'criterion': 'gini', 'm...",0.904290,1
711,"{'class_weight': None, 'criterion': 'gini', 'm...",0.904287,2
374,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
390,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
386,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
382,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
378,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
370,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
362,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
366,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3


## 5. Progress bar

In [10]:
n_estimators_list = [5, 10, 50, 100]
max_depth_list = list(range(1, 50))
class_weight_list = ['balanced', None]
criterion_list = ['entropy', 'gini']

all_combinations = list(product(n_estimators_list, max_depth_list, class_weight_list, criterion_list))

results = []
for n_est, depth, cw, crit in tqdm(all_combinations):
    rf = RandomForestClassifier(
        n_estimators=n_est, max_depth=depth,
        class_weight=cw, criterion=crit, random_state=21, n_jobs=-1
    )
    scores = cross_val_score(rf, X_train, y_train, cv=5, n_jobs=-1)
    results.append({
        'n_estimators': n_est,
        'max_depth': depth,
        'class_weight': cw,
        'criterion': crit,
        'mean_accuracy': scores.mean(),
        'std_accuracy': scores.std()
    })

results_df = pd.DataFrame(results).sort_values('mean_accuracy', ascending=False)
results_df.head(10)

  0%|          | 0/784 [00:00<?, ?it/s]

,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
503,50,28,None,gini,0.904290,0.010961
711,100,31,None,gini,0.904287,0.015204
569,50,45,balanced,gini,0.903549,0.012503
561,50,43,balanced,gini,0.903549,0.012503
565,50,44,balanced,gini,0.903549,0.012503
549,50,40,balanced,gini,0.903549,0.012503
573,50,46,balanced,gini,0.903549,0.012503
577,50,47,balanced,gini,0.903549,0.012503
581,50,48,balanced,gini,0.903549,0.012503
585,50,49,balanced,gini,0.903549,0.012503


## 6. Predictions

In [11]:
# Best model from gridsearch
best_rf = gs_rf.best_estimator_
best_rf.fit(X_train, y_train)
acc = accuracy_score(y_test, best_rf.predict(X_test))
print(f'Final accuracy on test: {acc:.5f}')

Final accuracy on test: 0.92899
